# LoRA Paper

Training Bottleneck: Updating a 175B parameter model requires storing the gradients and optimizer states (like Adam's momentum and variance) for all 175B parameters. This requires massive GPU clusters just for memory overhead.

Deployment Bottleneck: If you have 5 downstream tasks, full fine-tuning forces you to host 5 separate 175-billion-parameter models. That is a logistical and financial nightmare for inference.

# LoRA: Low-Rank Adaptation of Large Language Models (Hu et al., 2021)

## 1. Problem
*   **Compute & Memory:** Full fine-tuning is inherently doing the same work as training the full model on a specialized task. It requires updating and storing optimizer states for massive parameter matrices.
*   **Deployment Bottleneck:** If we have 5 downstream tasks, full fine-tuning forces us to host 5 separate, full-sized models. This scales horribly in production.

## 2. Solution (The "Intrinsic Dimension" Hypothesis)
*   **Observation:** The base model already possesses the core capabilities. It operates in a low "intrinsic dimension" when adapting to a specific task, meaning we do not need to utilize the full parameter space for specialized updates.
*   **Mechanism:** Freeze the pre-trained weight matrix ($W_0$) and approximate the update ($\Delta W$) using low-rank factorization.
*   **Formula:** $W = W_0 + BA$
    *   Where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times d}$
    *   Rank $r \ll \min(d, k)$

## 3. Advantages (Empirical Proof)
*   Using a layer where $d = 4096$ and $r = 8$:
    *   Full-rank update ($\Delta W$) parameters: $4096 \times 4096 = 16,777,216$
    *   LoRA update ($BA$) parameters: $B$ ($4096 \times 8$) + $A$ ($8 \times 4096$) = $65,536$
*   **Result:** The trainable parameters are reduced to 0.4% of the original parameter count. This eliminates massive memory overhead for optimizer states during training.

## 4. Questions
*   1. Knowledge Collapse (Catastrophic Forgetting)You are referring to the phenomenon where a model learns a new task but completely overwrites its general pre-trained knowledge.Does LoRA cause this?Structurally, true knowledge collapse of the base model is impossible with LoRA.Because we enforce W.requires_grad = False, the pre-trained weights ($W_0$) are mathematically protected. If your new task completely ruins the model's outputs, you simply delete the $BA$ matrices from memory, and your original model is 100% intact.However, there is a catch: Over-scaling the adapter.During the forward pass, $y = x(W_0 + BA)$. If you train the LoRA matrices for too long or with a learning rate that is too high, the values inside $BA$ can explode. When $BA$ becomes much larger than $W_0$ in magnitude, the update dominates the original weights, and the model's outputs will collapse into predicting only the new task.
*   2. Applicability to Computer Vision (CV) and Reinforcement Learning (RL)The LoRA formula ($\Delta W = BA$) does not care about text. It only cares about matrix multiplication. If an architecture uses dense linear layers, LoRA works.Computer Vision: LoRA is an industry standard here. Vision Transformers (ViTs) use exactly the same $W_q, W_k, W_v$ projection matrices as text transformers. Furthermore, image generation models like Stable Diffusion rely heavily on LoRA. Instead of full fine-tuning a 2-billion parameter diffusion model to learn a specific art style or face, ML engineers train a tiny LoRA on the cross-attention layers.Reinforcement Learning: Yes. As RL moves toward massive neural networks (like Decision Transformers or large Actor-Critic MLP networks), full fine-tuning a policy network for a new environment becomes too expensive. You can freeze the pre-trained policy ($W_0$) and train a LoRA adapter ($BA$) to adjust the agent's behavior for new physics or reward structures.